# 15 — The discriminating experiment

This is the moment the whole course has been building toward. In `14` you opened a second channel in the joint state $\rho_{AB}$ — the second circular moment of the phase difference, a quantity the phase-locking value cannot see. Today you turn that structural opening into a *measured* advantage: you build two systems that PLV is physically unable to tell apart, and watch a quantum-optimal-transport coupling measure separate them anyway.

**Prerequisites:** `14_richer_embedding_applied` (the multi-frequency qutrit, whose $\rho_{AB}$ carries the second moment in an element independent of PLV), and `13_the_redundancy_reveal` (the diagnosis this notebook resolves — on the naive embedding the quantum measure could only echo PLV).

**What you'll be able to do**
- Construct two ensembles with **identical PLV by construction** — same first circular moment $a_1$ — but **different second moment** $a_2$, using `matched_plv_ensembles`.
- Confirm the two ways of seeing only the first moment are both blind here: PLV is matched, and so is the **naive** phase-qubit embedding's quantum mutual information.
- Show the **richer multi-frequency embedding's** quantum mutual information **separates** the two ensembles — it reads the second moment PLV cannot.
- Establish that the separation is **real, not Monte-Carlo noise**, with an $a_2$-matched control: when both ensembles share $a_2$, the richer measure does *not* separate them, and the observed gap sits far above that null across many seeds.

This is the demonstration the restructure exists to deliver: a concrete, controlled case where a quantum-optimal-transport coupling measure beats PLV. We will state exactly what it shows — and, with equal care, what it does not.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qot_course import viz
from qot_course.colors import COLORS
from qot_course.quantum_ot.capstone import (
    plv,
    joint_density_matrix,
    coupling_qmi,
)
from qot_course.quantum_ot.embeddings import (
    multifreq_state,
    joint_density_from_states,
)
from qot_course.quantum_ot.discrimination import matched_plv_ensembles

np.random.seed(42)
viz.use_course_style()


def rich_qmi(theta_a, theta_b):
    """Quantum mutual information of the multi-frequency (qutrit) joint embedding.

    Embeds each phase record with multifreq_state(harmonics=(1, 2)) -> a 3-level state
    carrying the first AND second circular moments, joins them into the 9x9 rho_AB, and
    returns I(A:B) in nats. This is the 'richer-embedding QOT coupling measure' the
    experiment puts up against PLV.
    """
    rho = joint_density_from_states(multifreq_state(theta_a), multifreq_state(theta_b))
    return coupling_qmi(rho, dims=(3, 3))


# Sample count per channel. Large enough that PLV and the circular moments are tight
# Monte-Carlo estimates (error ~ 1/sqrt(n)), small enough that the notebook stays fast.
N = 150_000

## 1. The setup: two systems PLV cannot tell apart

Notebook `14` left us with a tool and a promise. The tool: the multi-frequency qutrit, whose joint density $\rho_{AB}$ carries the phase difference's **second** circular moment $\langle e^{2i\delta}\rangle$ in an element — $\rho_{AB}[6,2]$ — that the naive phase qubit does not even possess. The promise: that this second channel is an *independent* degree of freedom, so two couplings can share a PLV yet differ in it. Today we collect on that promise.

We build two ensembles, *low* and *high*, that are matched where PLV looks and parted where it cannot. Both draw their phase difference $\delta = \theta_A - \theta_B$ from the truncated-Fourier circular density

$$ p(\delta) \;\propto\; 1 + 2\,a_1 \cos\delta + 2\,a_2 \cos 2\delta, \qquad \delta \in [0, 2\pi), $$

whose first two circular moments are *exactly* $\langle e^{i\delta}\rangle = a_1$ and $\langle e^{2i\delta}\rangle = a_2$ (Mardia & Jupp 2000, ch. 3: the order-$k$ Fourier coefficient of a wrapped density is its $k$-th circular moment). We give both ensembles the **same $a_1 = 0.4$** — so they share their first moment, hence their PLV — but **different second moments**: $a_2 = 0.0$ for *low* (no second-order structure) and $a_2 = 0.3$ for *high*. A single uniform reference phase $\theta_A$, shared by both, keeps each single-channel marginal near-uniform, so the two ensembles differ in **nothing but** their joint second-order structure.

This is the cleanest possible test of the thesis. If a measure separates these two ensembles, it must be reading something past the first moment — because that is the only thing we let differ.

In [ ]:
# Two ensembles: identical first moment a1 (-> identical PLV), different second moment a2.
# matched_plv_ensembles shares one uniform reference phase theta_A across both dyads and
# draws independent phase differences (seed -> theta_A, seed+1 -> low delta, seed+2 -> high).
(theta_a_low, theta_b_low), (theta_a_high, theta_b_high) = matched_plv_ensembles(
    N, a1=0.4, a2_low=0.0, a2_high=0.3, seed=0
)

# The reference phase theta_A is shared, so the two ensembles differ only through delta.
print(f"samples per channel       N = {N:,}")
print(f"shared reference phase    theta_A identical across ensembles? "
      f"{np.array_equal(theta_a_low, theta_a_high)}")
print()
print("                          a1 (first moment)   a2 (second moment)")
print(f"low  ensemble             {0.4:>10.2f}        {0.0:>14.2f}")
print(f"high ensemble             {0.4:>10.2f}        {0.3:>14.2f}")

**Read the output.** Two ensembles, each $N$ phase samples per channel, sharing one reference phase $\theta_A$ (the `True` confirms it) so that *only* the phase difference $\delta$ — and within it, *only* the second moment $a_2$ — distinguishes them. Both carry the same first moment $a_1 = 0.4$; the low ensemble has no second-order structure ($a_2 = 0$) while the high ensemble has $a_2 = 0.3$. Everything that follows interrogates this single, deliberate difference: we have engineered two systems that agree on the first circular moment and disagree on the second, and now we ask each measure in turn whether it can hear the disagreement.

## 2. PLV is matched — by construction

The first measure is the neuroscience standard, the phase-locking value $\mathrm{PLV} = |\langle e^{i(\theta_A - \theta_B)}\rangle| = |\langle e^{i\delta}\rangle|$ (Lachaux et al. 1999). PLV is, definitionally, the magnitude of the **first** circular moment of the phase difference. We built both ensembles with the same $a_1$, so PLV must report the same value for both — up to the Monte-Carlo wobble of a finite sample.

We compute it for each ensemble and read off the gap.

In [ ]:
plv_low = plv(theta_a_low, theta_b_low)
plv_high = plv(theta_a_high, theta_b_high)

print(f"PLV (low ensemble,  a2 = 0.0)   {plv_low:.4f}")
print(f"PLV (high ensemble, a2 = 0.3)   {plv_high:.4f}")
print(f"PLV gap                         {abs(plv_low - plv_high):.4f}")

**Read the output.** The two PLV values agree to a few thousandths — a gap far below any threshold a phase-synchrony study would call a difference, and shrinking toward zero as $N$ grows (the residual is pure $1/\sqrt{N}$ Monte-Carlo wobble, since both ensembles share $a_1$ exactly in the limit). **By construction, PLV cannot distinguish these two systems.** To the phase-locking value they are the same dyad. If the second-order difference we built in is to be detected at all, it must be by a measure that looks past the first moment — and the rest of the notebook asks which measures can.

## 3. The naive quantum embedding is blind too

Before celebrating any quantum measure, we must rule out the obvious worry that *moving to a quantum state already buys the advantage*. It does not — and `13` told us why. The naive phase qubit $(|0\rangle + e^{i\theta}|1\rangle)/\sqrt 2$ carries only the first harmonic $e^{i\theta}$, so the single coupling-bearing coherence of its joint $\rho_{AB}$ is the first circular moment — the very quantity that *is* PLV. A quantum measure reading that state inherits PLV's blindness exactly.

So we compute the quantum mutual information of the **naive** $4\times 4$ joint density for each ensemble. If `13`'s diagnosis is right, the naive QMI must be matched here too — the redundancy of `13`, made concrete on these two specific systems.

In [ ]:
naive_qmi_low = coupling_qmi(joint_density_matrix(theta_a_low, theta_b_low))
naive_qmi_high = coupling_qmi(joint_density_matrix(theta_a_high, theta_b_high))

print("naive phase-qubit embedding  (4x4 joint density, QMI in nats)")
print(f"  QMI (low ensemble,  a2 = 0.0)   {naive_qmi_low:.5f}")
print(f"  QMI (high ensemble, a2 = 0.3)   {naive_qmi_high:.5f}")
print(f"  QMI gap                         {abs(naive_qmi_low - naive_qmi_high):.5f}")

**Read the output.** The naive embedding's quantum mutual information is matched between the two ensembles — its gap sits in the thousandths, the same Monte-Carlo floor we saw for PLV. This is exactly the redundancy `13` diagnosed, now playing out on these two engineered systems: because the naive qubit's only coupling channel *is* the first circular moment, the naive QMI is a re-encoding of PLV and inherits its blindness to $a_2$. Switching from a classical phase statistic to a quantum state, on its own, buys nothing here. The lesson of `13` stands: the bottleneck was the embedding, not the quantum measure — so the advantage, if it comes, must come from the **richer** embedding, not from the word *quantum*.

## 4. The richer embedding separates them

Now the measure built for the job. We embed each phase record with the multi-frequency qutrit $\tfrac{1}{\sqrt 3}\bigl(|0\rangle + e^{i\theta}|1\rangle + e^{2i\theta}|2\rangle\bigr)$ from `14`, join the two qutrits into the $9\times 9$ density $\rho_{AB}$, and take its quantum mutual information (this is the `rich_qmi` helper from the imports). The qutrit carries the second harmonic $e^{2i\theta}$, so its $\rho_{AB}$ holds the second circular moment $\langle e^{2i\delta}\rangle$ in the coherence $\rho_{AB}[6,2]$ — the channel the naive qubit lacked. The low and high ensembles differ precisely in that moment. If the richer-embedding QMI is doing what `14` promised, the two values should now come apart.

In [ ]:
rich_qmi_low = rich_qmi(theta_a_low, theta_b_low)
rich_qmi_high = rich_qmi(theta_a_high, theta_b_high)
rich_gap = abs(rich_qmi_low - rich_qmi_high)

print("richer multi-frequency embedding  (9x9 qutrit joint density, QMI in nats)")
print(f"  QMI (low ensemble,  a2 = 0.0)   {rich_qmi_low:.5f}")
print(f"  QMI (high ensemble, a2 = 0.3)   {rich_qmi_high:.5f}")
print(f"  QMI gap                         {rich_gap:.5f}")
print()
print("for contrast, the gaps the blind measures reported:")
print(f"  PLV gap                         {abs(plv_low - plv_high):.5f}")
print(f"  naive-embedding QMI gap         {abs(naive_qmi_low - naive_qmi_high):.5f}")

**Read the output.** The richer embedding's quantum mutual information is **different** for the two ensembles: the high ensemble ($a_2 = 0.3$) carries more $A$–$B$ information than the low ($a_2 = 0.0$), and the gap is several times larger than the matched gaps PLV and the naive embedding reported. Stacking the three rows tells the whole story at a glance — PLV's gap and the naive-QMI gap sit at the Monte-Carlo floor (thousandths), while the rich-QMI gap stands clearly above it. The richer-embedding QOT coupling measure is reading structure that PLV — and the naive quantum embedding — physically cannot see. That is the discrimination the course set out to demonstrate.

One honest caveat before we trust it. A single pair of numbers being different is not yet evidence: any two finite samples differ a little. The gap above is small in absolute terms, and we have only looked at one draw. The next section earns the claim — it shows that this gap is *caused by* $a_2$, by switching $a_2$ off as a control and checking that the separation vanishes, and that the observed gap stands far above that null across many seeds.

## 5. Significance: the separation is real, not noise

The clincher is a control. The thesis is that the gap in section 4 is driven by the **difference in $a_2$** — not by the qutrit embedding inventing structure, and not by Monte-Carlo luck. The way to prove it is to remove the cause and check the effect disappears: build pairs of ensembles whose $a_2$ is **matched** (both $0.0$), so they differ in *nothing* but sampling noise, and measure the rich-QMI gap on each. That is the **$a_2$-matched null distribution** — the spread of gaps you get from the richer measure when there is genuinely nothing to find.

We run that null across many seeds, and beside it the $a_2$-**differs** condition (low $a_2=0$ vs high $a_2=0.3$) across the same number of seeds, so we can compare two whole distributions rather than two lone numbers. The thesis predicts the differs distribution sits far above the matched null. (We use independent master seeds spaced by 10, since `matched_plv_ensembles` consumes `seed`, `seed+1`, `seed+2` internally.)

In [ ]:
n_seeds = 12  # a dozen independent runs per condition -> two gap distributions to compare

null_gaps = np.empty(n_seeds)   # a2 MATCHED (0.0 vs 0.0): nothing to find -> the null
diff_gaps = np.empty(n_seeds)   # a2 DIFFERS (0.0 vs 0.3): the real second-moment difference

for i in range(n_seeds):
    # a2-matched control: both ensembles a2 = 0.0, so any gap is pure Monte-Carlo noise.
    (na, nb), (nc, nd) = matched_plv_ensembles(
        N, a1=0.4, a2_low=0.0, a2_high=0.0, seed=1000 + 10 * i
    )
    null_gaps[i] = abs(rich_qmi(na, nb) - rich_qmi(nc, nd))

    # a2-differs: the experiment's condition, across fresh seeds.
    (da, db), (dc, dd) = matched_plv_ensembles(
        N, a1=0.4, a2_low=0.0, a2_high=0.3, seed=5000 + 10 * i
    )
    diff_gaps[i] = abs(rich_qmi(da, db) - rich_qmi(dc, dd))

print(f"rich-QMI gap over {n_seeds} seeds per condition (nats)")
print()
print(f"a2-MATCHED null (0.0 vs 0.0)   mean {null_gaps.mean():.5f}   "
      f"min {null_gaps.min():.5f}   max {null_gaps.max():.5f}")
print(f"a2-DIFFERS      (0.0 vs 0.3)   mean {diff_gaps.mean():.5f}   "
      f"min {diff_gaps.min():.5f}   max {diff_gaps.max():.5f}")
print()
# Does the smallest differs-gap still exceed the largest null-gap? (clean separation)
print(f"smallest differs gap exceeds largest null gap?  "
      f"{diff_gaps.min() > null_gaps.max()}")
# How many null standard deviations above the null mean does the observed gap sit?
z = (rich_gap - null_gaps.mean()) / null_gaps.std()
print(f"observed section-4 gap sits {z:.1f} null-std above the null mean")

**Read the output.** Two distributions of the rich-QMI gap. The $a_2$-matched null — both ensembles with $a_2 = 0.0$, so nothing distinguishes them but sampling — clusters near a small mean with a modest spread: this is the measure's noise floor, the gap it reports when there is truly nothing to find. The $a_2$-differs condition clusters several times higher. The two do not overlap — the *smallest* differs gap still exceeds the *largest* null gap (the `True`), and the observed section-4 gap sits many null standard deviations above the null mean. The separation is not Monte-Carlo luck: switch $a_2$ off and it collapses into the noise; switch it on and it stands well clear. Let us see the two clouds side by side.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

# Jittered strips so individual seeds are visible, not only summary statistics.
rng_jit = np.random.default_rng(0)
x_null = 0.0 + rng_jit.uniform(-0.06, 0.06, size=n_seeds)
x_diff = 1.0 + rng_jit.uniform(-0.06, 0.06, size=n_seeds)

ax.scatter(
    x_null, null_gaps, s=90, color=COLORS["muted"], alpha=0.85,
    edgecolor="white", linewidth=0.8, zorder=3,
    label=r"$a_2$ matched (0.0 vs 0.0) — the null",
)
ax.scatter(
    x_diff, diff_gaps, s=90, color=COLORS["highlight"], alpha=0.9,
    edgecolor="white", linewidth=0.8, zorder=3,
    label=r"$a_2$ differs (0.0 vs 0.3) — the experiment",
)

# The null's ceiling and the observed section-4 gap, as reference lines.
ax.axhline(
    null_gaps.max(), color=COLORS["muted"], lw=1.4, ls="--", zorder=2,
    label="largest null gap (noise ceiling)",
)
ax.axhline(
    rich_gap, color=COLORS["quantum"], lw=2.0, ls="-", zorder=2,
    label="observed section-4 gap",
)

ax.set_xticks([0.0, 1.0])
ax.set_xticklabels(["matched $a_2$\n(null)", "differs $a_2$\n(experiment)"])
ax.set_ylabel("rich-QMI gap between the two ensembles  (nats)")
ax.set_ylim(bottom=0.0)
ax.set_title("The second-moment separation stands clear of its matched-$a_2$ null", pad=12)
ax.legend(loc="upper left", fontsize=9)
fig.tight_layout()
plt.show()

**Read the figure.** Two columns of dots, one per condition, each dot a single seed's rich-QMI gap. The grey column on the left is the $a_2$-matched null: every gap small, all bunched near the floor, because those ensemble pairs differ in nothing but noise. The rose column on the right is the $a_2$-differs experiment: every gap clearly higher, the whole cloud lifted above the grey one. The dashed grey line marks the highest gap the null ever produced — the noise ceiling — and **every** rose dot sits above it; the two distributions do not touch. The solid blue line marks the single observed gap from section 4, sitting comfortably inside the rose cloud and far above the null.

This is what "real, not noise" looks like made visible. When $a_2$ is matched, the richer measure correctly reports near-nothing — it is not hallucinating structure. When $a_2$ differs, it reports a gap that no amount of sampling noise in the null ever reaches. The separation tracks the second moment we built in, and only that. The richer-embedding QOT coupling measure has detected, with a control, a difference PLV is blind to.

## Your turn

**Warm-up.** Section 4 used $a_2 = 0.3$ for the high ensemble. Predict what happens to the rich-QMI gap if you lower the high ensemble's second moment toward the low one — say to $a_2 = 0.1$, then $a_2 = 0.05$. Which direction does the gap move, and what does it approach as $a_2^{\text{high}} \to a_2^{\text{low}}$? Relate your prediction to the fact that the gap is driven by the *difference* in second moments, not by either value alone.

**Go further.** The control in section 5 set both ensembles to $a_2 = 0.0$. Predict the rich-QMI gap distribution if you instead match them at $a_2 = 0.3$ (both high). Should the null look the same, larger, or smaller than the $a_2 = 0.0$ null — and why does *matching* the second moment, at whatever value, send the separation back into the noise? State what this second null would add to the argument that $a_2$ is the cause.

**Challenge.** PLV is matched, the naive quantum embedding is matched, and the richer quantum embedding separates the ensembles. Argue, in words, why this does **not** license the claim that *only* a quantum measure could detect the difference. Name a purely classical statistic of the phase difference that would also tell the two ensembles apart, and explain what genuinely *is* the achievement here — what the quantum-optimal-transport framework, handed a richer embedding, gives you that a hand-picked classical statistic does not. (You are previewing the honest accounting of `16`.)

## What you built

This is the payoff of the entire course. You demonstrated, with a control, a concrete case where a quantum-optimal-transport coupling measure **beats the phase-locking value**.

- You constructed two ensembles with **identical PLV by construction** — same first circular moment $a_1 = 0.4$ — but **different second moment** ($a_2 = 0.0$ vs $0.3$), differing in nothing else, using `matched_plv_ensembles`.
- You confirmed both first-moment views are blind to the difference: **PLV is matched**, and the **naive phase-qubit embedding's QMI is matched** too — the redundancy of `13`, made concrete on these two systems.
- You showed the **richer multi-frequency embedding's QMI separates** them, with a gap several times the matched floor — it reads the second circular moment the others cannot.
- You established the separation is **real, not noise**: with $a_2$ matched (the null) the richer measure does not separate, and across many seeds the $a_2$-differs gap stands entirely above that null, the observed gap many null-std above its mean.

State the result precisely, and no further: the richer-embedding QOT coupling measure separates two ensembles PLV cannot — *not* that only a quantum measure could, since a classical second-moment statistic would also see $a_2$. The achievement is that the quantum-optimal-transport framework, given a richer embedding, **naturally** accesses higher-order coupling structure rather than needing a moment hand-picked in advance.

That "naturally" carries a cost as well as a benefit, and an honest demonstration owes you the full accounting. In `16_capstone_synthesis` we close the arc: when the richer embedding is worth its added dimensions, the statistical price of the extra channels, and the caveat — that a bespoke classical statistic can chase any single moment — laid out plainly beside today's win.

## References

- J.-P. Lachaux, E. Rodriguez, J. Martinerie & F. J. Varela, "Measuring phase synchrony in brain signals", *Human Brain Mapping* **8**, 194–208 (1999). DOI:10.1002/(SICI)1097-0193(1999)8:4<194::AID-HBM4>3.0.CO;2-C — the phase-locking value $\mathrm{PLV} = |\langle e^{i\Delta\theta}\rangle|$, the first circular moment that is matched across the two ensembles by construction.
- K. V. Mardia & P. E. Jupp, *Directional Statistics* (Wiley, 2000). DOI:10.1002/9780470316979 — circular (trigonometric) moments of a phase distribution; the first and second moments are distinct functionals, which is what lets us match PLV while parting the second moment.
- D. Trevisan, "Optimal transport methods for quantum systems", arXiv:2202.02091 (2022). DOI:10.48550/arXiv.2202.02091 — quantum-transport coupling measures (quantum mutual information, Bures) that read the enriched $\rho_{AB}$ and deliver the separation.

**Previous:** `notebooks/04_QuantumOptimalTransport/14_richer_embedding_applied.ipynb` · **Next:** `notebooks/04_QuantumOptimalTransport/16_capstone_synthesis.ipynb`